In [0]:
orders = spark.table("workspace.default.silver_orders")
order_items = spark.table("workspace.default.silver_order_items")
payments = spark.table("workspace.default.silver_order_payments")
reviews = spark.table("workspace.default.silver_order_reviews")
display(orders)
display(order_items)
display(payments)
display(reviews)

In [0]:
payments.printSchema()

In [0]:
from pyspark.sql.functions import col

In [0]:
gold_sales = (
    orders.alias("o")
    .join(
        order_items.alias("oi"),
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )
   .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("o.order_status"),
        col("o.order_purchase_timestamp"),
        col("o.order_delivered_customer_date"),
        col("oi.order_item_id"),
        col("oi.product_id"),
        col("oi.seller_id"),
        col("oi.shipping_limit_date"),
        col("oi.price"),
        col("oi.freight_value")
)
)


In [0]:
display(gold_sales)

In [0]:
gold_sales = (
    gold_sales.alias("g")
    .join(
        payments.alias("p"),
        col("g.order_id") == col("p.order_id"),
        "inner"
    )
    .select(
        col("g.*"),
        col("p.payment_sequential"),
        col("p.payment_type"),
        col("p.payment_installments"),
        col("p.payment_value")
    )
)

In [0]:
display(gold_sales)

In [0]:
gold_sales = (
    gold_sales.alias("g")
    .join(
        reviews.alias("r"),
        col("g.order_id") == col("r.order_id"),
        "left"
    )
    .select(
        col("g.*"),
        col("r.review_id"),
        col("r.review_score"),
        col("r.review_creation_date"),
        col("r.review_answer_timestamp")
    )
)

In [0]:
display(gold_sales)

In [0]:
gold_sales.printSchema()

In [0]:
print("Total Rows: ", gold_sales.count())
print("Total Columns: ", len(gold_sales.columns))

In [0]:
gold_sales.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_sales")

In [0]:
display(spark.table("workspace.default.gold_sales"))

In [0]:
customers = spark.table("workspace.default.silver_customers")

In [0]:
customers.printSchema()


In [0]:
gold_customers = (
    customers.select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
)

In [0]:
gold_customers.printSchema()

display(gold_customers)

In [0]:
print("Total Rows: ", gold_customers.count())
print("Total Columns: " , len(gold_customers.columns))

In [0]:
gold_customers.write.mode("overwrite").saveAsTable("workspace.default.gold_customers")

In [0]:
display(spark.table("workspace.default.gold_customers"))

In [0]:
products = spark.table("workspace.default.silver_products")
categories = spark.table("workspace.default.silver_category_translation")

In [0]:
products.printSchema()

In [0]:
from pyspark.sql.functions import col 
gold_products = (
    products.alias("p")
    .join(
        categories.alias("c"),
        col("p.product_category_name") == col("c.product_category_name"),
        "left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col("c.product_category_name_english"),
        col("p.product_name_lenght"),
        col("p.product_description_lenght"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm")
    )
)

In [0]:
gold_products.printSchema()
display(gold_products)

In [0]:
print("Total Rows: ", gold_products.count())
print("Total Columns: ", len(gold_products.columns))

In [0]:
gold_products.write.mode("overwrite").saveAsTable("workspace.default.gold_products")

In [0]:
display(spark.table("workspace.default.gold_products"))

In [0]:
sellers = spark.table("workspace.default.silver_sellers")

In [0]:
sellers.printSchema()

In [0]:
gold_sellers = (
    sellers.select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
)

In [0]:
display(gold_sellers)

In [0]:
print("Total Rows: " , gold_sellers.count())
print("Total Columns: " , len(gold_sellers.columns))

In [0]:
gold_sellers.write.mode("overwrite").saveAsTable("workspace.default.gold_sellers")

In [0]:
display(spark.table("workspace.default.gold_sellers"))

In [0]:
delivery = spark.table("workspace.default.silver_orders")

In [0]:
delivery.printSchema()

In [0]:
from pyspark.sql.functions import col , datediff , when

In [0]:
gold_delivery = (
    delivery.select(
        "order_id",
        "order_purchase_timestamp" , 
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
    .withColumn(
        "delivery_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "estimated_delivery_days",
        datediff(
            col("order_estimated_delivery_date"),
            col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_status",
        when(
            col("order_delivered_customer_date") <= col("order_estimated_delivery_date") , "On Time"
        ).otherwise("Delayed")
    )
)

In [0]:
display(gold_delivery)

In [0]:
print("Total Rows: " , gold_delivery.count())
print("Total Columns: ", len(gold_delivery.columns))

In [0]:
gold_delivery.write.mode("overwrite").saveAsTable("workspace.default.gold_delivery")

In [0]:
display(spark.table("workspace.default.gold_delivery"))